# 02 · Giao hàng, nhóm ưu tiên, mua lại và khoảng cách
Nguồn duy nhất: các view `mart.vw_logistics`, `vw_seller`, `vw_category`, `vw_customer`. SQL đã tính eligible, is_late, thời gian các chặng, distance_km và repeat_90d. Python chỉ tổng hợp kết quả, trực quan hóa và tính thống kê. Chạy SQL 08 trước; **Restart Kernel → Run All**.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sql/07_create_mart_views.sql').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project or notebooks folder.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from utils.db import get_engine, read_mart
START, END = pd.Timestamp('2017-01-01'), pd.Timestamp('2018-09-01')
EXPORT_RESULTS = False  # Set True to save the displayed tables and figures.
OUT = ROOT / 'data/exports/sql_mart'
tables, figures = {}, {}
def cohort(frame):
    return frame.loc[frame.order_purchase_timestamp.ge(START)
                     & frame.order_purchase_timestamp.lt(END)].copy()

In [ ]:
engine = get_engine()
try:
    orders = cohort(read_mart('logistics', engine))
    sellers = cohort(read_mart('seller', engine))
    categories = cohort(read_mart('category', engine))
    first_orders = cohort(read_mart('customer', engine))
finally:
    engine.dispose()
assert orders.order_id.is_unique and not orders.empty
assert not sellers.duplicated(['order_id', 'seller_id']).any()
assert not categories.duplicated(['order_id', 'category_en']).any()
assert first_orders.customer_unique_id.is_unique
eligible = orders[orders.is_eligible.eq(1)].copy()
assert eligible.is_late.isin([0, 1]).all() and not eligible.empty

## SLA theo tháng và thời gian các chặng
Late rate = số đơn trễ / eligible. Duration âm được SQL để NULL; COUNT(duration) là cỡ mẫu của từng chặng. P90 của các chặng không cộng thành P90 toàn hành trình.

In [ ]:
monthly = eligible.groupby('order_month').agg(
    eligible_orders=('order_id', 'size'), late_orders=('is_late', 'sum'),
    late_rate=('is_late', 'mean'), delivery_sample=('delivery_days', 'count'),
    median_days=('delivery_days', 'median'), p90_days=('delivery_days', lambda s: s.quantile(.9)),
    median_delay=('delay_days', 'median'), p90_delay=('delay_days', lambda s: s.quantile(.9))).reset_index()
stages = pd.DataFrame([{'stage': col, 'sample': eligible[col].count(),
    'median_days': eligible[col].median(), 'p90_days': eligible[col].quantile(.9)}
    for col in ['approval_days', 'handoff_days', 'transit_days']])
tables.update(delivery_monthly=monthly, delivery_stages=stages)
display(monthly); display(stages)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(monthly.order_month, monthly.late_rate * 100)
axes[0].set(title='Late rate by purchase month', ylabel='% eligible orders')
axes[1].plot(monthly.order_month, monthly.median_days, label='Median')
axes[1].plot(monthly.order_month, monthly.p90_days, label='P90')
axes[1].set(ylabel='Days', title='Actual delivery time'); axes[1].legend()
fig.autofmt_xdate(); fig.tight_layout(); plt.show(); figures['delivery_monthly'] = fig

## Shortlist theo số đơn trễ, ngưỡng 30/50/100
Seller và category có thể chồng lấp giữa các nhóm. Route được khử trùng theo order + seller_state + customer_state vì nhiều seller có thể thuộc cùng tuyến. Outcome cấp đơn không chứng minh seller nào gây trễ.

In [ ]:
seller_eligible = sellers[sellers.is_eligible.eq(1)].copy()
category_eligible = categories[categories.is_eligible.eq(1)].copy()
routes = seller_eligible.drop_duplicates(['order_id', 'seller_state', 'customer_state'])
def priorities(frame, groups):
    result = frame.groupby(groups, dropna=False).agg(
        eligible_orders=('order_id', 'size'), late_orders=('is_late', 'sum'),
        late_rate=('is_late', 'mean'), median_days=('delivery_days', 'median'),
        p90_days=('delivery_days', lambda s: s.quantile(.9))).reset_index()
    return pd.concat([result[result.eligible_orders.ge(n)].sort_values(
        ['late_orders', *groups], ascending=[False] + [True]*len(groups)).assign(minimum_orders=n)
        for n in [30, 50, 100]], ignore_index=True)
for name, frame, groups in [('states', eligible, ['customer_state']),
    ('categories', category_eligible, ['category_en']), ('sellers', seller_eligible, ['seller_id']),
    ('routes', routes, ['seller_state', 'customer_state'])]:
    table = priorities(frame, groups)
    tables['priority_' + name] = table
    print(name); display(table[table.minimum_orders.eq(30)].head(10))
threshold_comparison = pd.DataFrame([{'segment': name, 'minimum_orders': n,
    'qualified_groups': len(tables['priority_' + name].query('minimum_orders == @n'))}
    for name in ['states', 'categories', 'sellers', 'routes'] for n in [30, 50, 100]])
tables['threshold_comparison'] = threshold_comparison
display(threshold_comparison)

## Mua lại trong 90 ngày sau đơn đầu
SQL chọn đơn đầu trên toàn lịch sử trước khi notebook lọc cohort. Chỉ giữ đơn đầu eligible và has_90d_followup=1; mốc cuối quan sát trong view là 2018-09-01. Wilson 95% thể hiện độ bất định, không chứng minh nhân quả. Không tính lại repeat bằng join CSV.

In [ ]:
first = first_orders[first_orders.is_eligible.eq(1) & first_orders.has_90d_followup.eq(1)]
retention = first.groupby('is_late').agg(customers=('customer_unique_id', 'size'),
    repeat_customers=('repeat_90d', 'sum')).reset_index()
retention['repeat_rate'] = retention.repeat_customers / retention.customers
def wilson(k, n):
    if n == 0: return np.nan, np.nan
    p, z = k/n, 1.96
    denominator = 1 + z*z/n
    center = (p + z*z/(2*n))/denominator
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/denominator
    return center-half, center+half
retention['ci_low'] = [wilson(k,n)[0] for k,n in zip(retention.repeat_customers,retention.customers)]
retention['ci_high'] = [wilson(k,n)[1] for k,n in zip(retention.repeat_customers,retention.customers)]
tables['retention_90d'] = retention; display(retention)

## Khoảng cách và vận chuyển
distance_km do SQL tính từ centroid zip. Phân tầng tuyến theo khoảng cách; correlation dùng khoảng cách seller xa nhất mỗi đơn để một đơn không bị đếm nhiều lần. Đây là đường chim bay, không phải quãng đường thực tế.

In [ ]:
from scipy.stats import spearmanr
seller_eligible['distance_band'] = pd.cut(seller_eligible.distance_km,
    [-np.inf, 500, 1500, np.inf], labels=['<500', '500-1499', '1500+'], right=False
    ).astype('string').fillna('missing')
distance_routes = seller_eligible.groupby(['seller_state', 'customer_state', 'distance_band'],
    dropna=False, observed=True).agg(order_seller_pairs=('order_id', 'size'),
    transit_sample=('transit_days', 'count'), mean_transit=('transit_days', 'mean'),
    late_rate=('is_late', 'mean')).reset_index()
distance_orders = seller_eligible.groupby('order_id').agg(distance_km=('distance_km', 'max'),
    transit_days=('transit_days', 'first')).dropna()
rho = p_value = np.nan
if len(distance_orders) >= 3 and distance_orders.distance_km.nunique()>1 and distance_orders.transit_days.nunique()>1:
    rho, p_value = spearmanr(distance_orders.distance_km, distance_orders.transit_days)
distance_test = pd.DataFrame([{'orders':len(distance_orders), 'spearman_rho':rho, 'p_value':p_value}])
tables.update(distance_routes=distance_routes, distance_correlation=distance_test)
display(distance_routes.head(20)); display(distance_test)

In [ ]:
if EXPORT_RESULTS:
    OUT.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')
    for name, figure in figures.items():
        figure.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight')
    print(f'Exported SQL mart results to {OUT}')
else:
    print('Results displayed only. Set EXPORT_RESULTS=True and Run All to export.')